# Visualize Weka speed data

In [ ]:
# Imports
import datetime
import json

from pathlib import Path

from pydantic import BaseModel

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

from weka_helpers import (
    ResultMap,
    RatioContainer,
    Generation,
    Location,
    load_results_from_json,
)

In [ ]:
# Input parameters

input_dir = Path("/rubin/bench/weka/current")
input_files: list[Path] = input_dir.glob("*.json")

In [ ]:
type ResultMapDataframe = dict[str,list[int|float|str]]

def add_data_to_rmd(
    rmd: ResultMapDataframe,
    rm: ResultMap,
    generation: str,
    location:str
) -> ResultMapDataframe:
    idx = list(range(1,len(rm.sizes)+1))
        
    rmd["index"].extend(idx)
    rmd["urls"].extend(list(rm.urls))
    rmd["sizes"].extend(list(rm.sizes))
    rmd["start_times"].extend(list(rm.start_times))
    rmd["end_times"].extend(list(rm.end_times))
    rmd["elapsed_times"].extend([x.total_seconds() for x in rm.elapsed_times])
    rmd["speeds_reported"].extend(list(rm.speeds_reported))
    rmd["generation"].extend([generation for x in rm.sizes])
    rmd["location"].extend([location for x in rm.sizes])
    rmd["marker_size"].extend([0.2 for x in rm.sizes])
    return rmd

def ratio_container_to_dataframe(ratio_container: RatioContainer) -> pd.DataFrame:
    rmd = {
        "index": [],
        "urls": [],
        "sizes": [],
        "start_times": [],
        "end_times": [],
        "elapsed_times": [],
        "speeds_reported": [],
        "generation": [],
        "location": [],
        "marker_size": [],   
    }
    rmd=add_data_to_rmd(
        rmd,
        ratio_container.initial,
        generation="initial",
        location=ratio_container.initial.location
    )
    if ratio_container.recheck:
        rmd=add_data_to_rmd(
            rmd,
            ratio_container.recheck,
            generation="recheck",
            location=ratio_container.recheck.location
        )
    return pd.DataFrame(data=rmd)

In [ ]:
def plot_individual_speeds(ratio_container: RatioContainer) -> None:
    df = ratio_container_to_dataframe(ratio_container)
    max_speed = max(df.speeds_reported)
    labels = {
        "index": "index",
        "urls": "url",
        "sizes": "size (bytes)",
        "start_times": "start time (UTC)",
        "end_times": "end time (UTC)",
        "elapsed_times": "elapsed time",
        "speeds_reported": "speed (reported) (bytes/s)",
        "generation": "generation",
        "location": "location",
        "marker_size": "marker size",
    }    
    fig = px.bar(
        df,
        x="index",
        y="speeds_reported",
        color="generation",
        facet_row="generation",
        labels=labels,
        hover_data={
            "index": True,
            "urls": False,
            "sizes": True,
            "start_times": True,
            "end_times": True,
            "elapsed_times": True,
            "speeds_reported": True,
            "generation": True,
            "location": True,
            "marker_size": False,
        },        
        range_y=[0, 1.1 * max_speed],
        title = (
            "S3 Butler speed\n"
            f"{ratio_container.display_name}"
            f"[{ratio_container.dataset_name}@{ratio_container.initial.location}]"
        ),
        subtitle = (
            f"{ratio_container.count}/{ratio_container.limit} records\n"
            f"{ratio_container.user}@{ratio_container.node}"
        )
    )
    fig.show()
    del(fig)

In [ ]:
def aggregate_speeds_from_resultmap(
    resultmap: ResultMap,
) -> pd.DataFrame:
    speed_series = resultmap.aggregate_speeds_reported
    color=0 if resultmap.generation == Generation.INITIAL else 1
    return pd.DataFrame(
        {
            "time": list(speed_series.keys()),
            "speed": list(speed_series.values()),
        }
    )
    
def plot_aggregate_rm_speeds(
    df: pd.DataFrame,
    *,
    generation: Generation,
    location: Location,
    display_name: str,
    dataset_name: str,
    node: str,
    user: str,
    count: int,
    limit: int,
) -> None:
    max_speed = max(df.speed)
    color_seq = ["red"] if generation == generation.RECHECK else ["blue"]
    fig=px.line(
        df,
        x="time",
        y="speed",
        color_discrete_sequence=color_seq,
        hover_data={
            "time": True,
            "speed": True,
        },
        labels={
            "time": "time (UTC)",
            "speed": "speed (bytes/s)",
        },
        markers=True,
        range_y=[0, 1.1 * max_speed],
        title=(
            f"S3 Butler throughput"
            f"({generation})\n"
            f"{display_name}"
            f"[{dataset_name}@{location}] \n"
        ),
        subtitle=(
            f"{count}/{limit} records\n"        
            f"{user}@{node}"
        ),
    )
    fig.show()
    del(fig)
            
def plot_aggregate_speeds(ratio_container: RatioContainer) -> None:
    node=ratio_container.node
    display_name=ratio_container.display_name
    dataset_name=ratio_container.dataset_name
    count=ratio_container.count
    limit=ratio_container.limit
    user=ratio_container.user
    location=ratio_container.initial.location
    plot_aggregate_rm_speeds(
        aggregate_speeds_from_resultmap(
            ratio_container.initial
        ),
        generation=Generation.INITIAL,
        display_name=display_name,
        dataset_name=dataset_name,
        node=node,
        user=user,
        count=count,
        limit=limit,
        location=location,
    )
    if not ratio_container.recheck:
        return
    plot_aggregate_rm_speeds(
        aggregate_speeds_from_resultmap(
            ratio_container.recheck
        ),
        generation=Generation.RECHECK,
        display_name=display_name,
        dataset_name=dataset_name,
        node=node,
        user=user,
        count=count,
        limit=limit,
        location=location
    )
    plot_cache_ratios(
        ratios_from_container(
            ratio_container
        ),
        display_name=display_name,
        dataset_name=dataset_name,
        node=node,
        user=user,
        count=count,
        limit=limit,
        location=location
    )

In [ ]:
def ratios_from_container(
    resultmap: ResultMap,
) -> pd.DataFrame:
    ratio_series = resultmap.ratios_reported
    idx = [ 1 + x[0] for x in enumerate(ratio_series) ]
    return pd.DataFrame(
        {
            "index": idx,
            "ratio": list(ratio_series)
        }
    )
    
def plot_cache_ratios(
    df: pd.DataFrame,
    *,
    display_name: str,
    dataset_name: str,
    node: str,
    user: str,
    count: int,
    limit: int,
    location: str,
) -> None:
    max_ratio=max(df.ratio)
    fig=px.line(
        df,
        x="index",
        y="ratio",
        color_discrete_sequence=["violet"],
        hover_data={
            "index": True,
            "ratio": True,
        },
        labels={
            "index": "sample #",
            "ratio": "recheck/initial speed",
        },
        markers=True,
        range_y=[0, 1.1 * max_ratio],
        title=(
            "S3 Butler recheck/initial speed ratio \n"
            f"{display_name}"
            f"[{dataset_name}@{location}] \n"
        ),
        subtitle=(
            f"{count}/{limit} records\n"        
            f"{user}@{node}"
        ),
    )
    fig.show()
    del(fig)

# Plot results

In [ ]:
def plot_result(ratio_container: RatioContainer) -> None:
    plot_individual_speeds(ratio_container)
    plot_aggregate_speeds(ratio_container)

res: list[RatioContainer] = []
for fn in input_files:
    rms=load_results_from_json(fn)
    res.extend(rms)

for ratio_container in res:
    plot_result(ratio_container)


In [ ]:
coadd_google=res[0]
visit_usdf=res[1]

print("initial ratio Google/USDF:"
        f" {(coadd_google.initial.average_speed/visit_usdf.initial.average_speed):.02f}\n"
        "recheck ratio Google/USDF:"
        f" {(coadd_google.recheck.average_speed/visit_usdf.recheck.average_speed):.02f}")

In [ ]:
print(f"Google/USDF mean reported speed ({coadd_google.initial.statistics.reported.mean:.02f}):"
    f" {(coadd_google.initial.statistics.reported.mean / visit_usdf.initial.statistics.reported.mean):.02f}\n"
    f"Google/USDF median reported speed: ({coadd_google.initial.statistics.reported.median:.02f})"
    f" {(coadd_google.initial.statistics.reported.median / visit_usdf.initial.statistics.reported.median):.02f}")
